## Grant Deduplication (S2)

Deduplicates newly-queried Dimensions grants (from S1) against tracked/historical grant sources - last year's curated data, the GFI Grants Tracker (Airtable export), and `funding_inscope` - before the survivors go to S3 for LLM scope screening.

This is a manual notebook, not an automated script: four steps pause for you to review and confirm full/partial title matches in an exported CSV before continuing. Redesigned from `SHEEP/Funding/grant_deduplication.ipynb` - see that notebook for the original prototype and its "Stella" validation notes.

**Before running:** set `RUN_TABLE` in the config cell below to the run you want to process - the name `pipeline_funding.py` printed when it asked you to run this notebook (e.g. `run_260721_1400`).

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

import duckdb
import pandas as pd
import numpy as np
import pycountry
import re

sys.path.append(str(Path.cwd()))
from Funding_dedup_helpers import (
    is_empty, is_zero, normalize_title, assign_stable_row_id,
    gap_fill, gap_fill_researchers_and_orgs,
    export_highlighted_diff, export_for_review, apply_reviewed_decisions,
)

# CONFIG - edit before running
RUN_TABLE = 'run_YYMMDD_HHMM'  # <-- set this to match the run printed by pipeline_funding.py

REPORT_YEAR = 2026        # this year's curated/ground-truth report (funding_inscope)
PREV_REPORT_YEAR = 2025   # previous year's curated data (last_year_data)

DB_PATH = Path('../../Funding/funding.db')             # kept in place, not duplicated here
RAW_DATA_DIR = Path('../../Funding/1_deduplication/raw_data')
DATA_DIR = Path('data')            # review CSVs (export_for_review / apply_reviewed_decisions)
AUDIT_DIR = Path('data_output')    # highlighted-diff audit exports + final table Excel mirrors
DATA_DIR.mkdir(exist_ok=True)
AUDIT_DIR.mkdir(exist_ok=True)

FUNDING_INSCOPE_FILE = f'Funding{REPORT_YEAR}_inscope.xlsx'
LAST_YEAR_FILE = f'Funding{PREV_REPORT_YEAR}_inscope.xlsx'
GRANTS_TRACKER_FILE = 'GrantsTracker_2026-06-30.xlsx'  # update to the latest tracker export as needed

FUZZY_THRESHOLD = 85  # 0-100; rapidfuzz token_sort_ratio threshold for partial title matches

DEDUP_TABLE = f'{RUN_TABLE}_dedup'
CURATED_TABLE = f'funding_curated_{REPORT_YEAR}'
RUN_DATE = datetime.today().strftime('%y%m%d')

print(f"RUN_TABLE = {RUN_TABLE}")
print(f"DEDUP_TABLE (output, S3 reads this) = {DEDUP_TABLE}")
print(f"CURATED_TABLE (output, archival)     = {CURATED_TABLE}")

### 0. Ground truth: this year's curated data (`funding_inscope`)

Not used for matching in this notebook - it's the read-only ground-truth reference the LLM scope-screening evaluation work relies on. Kept here since it shares a source format with the previous-year import below.

In [ ]:
def clean_dtypes(df, has_duration_years):
    df = df.copy()
    datetime_cols = [
        'Date request submitted', 'Date award announced',
        'Project start date', 'Date added', 'Last modified',
    ]
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').astype('datetime64[us]')

    year_cols = ['Year request submitted', 'Year project started', 'End date']
    for col in year_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    def _clean_months(x):
        if pd.isna(x) or isinstance(x, pd.Timestamp):
            return None
        try:
            return int(float(x))
        except (ValueError, TypeError):
            return None

    if 'Duration of award (months)' in df.columns:
        df['Duration of award (months)'] = df['Duration of award (months)'].apply(_clean_months).astype('Int64')

    if has_duration_years and 'duration (years)' in df.columns:
        def _clean_years(x):
            if pd.isna(x) or isinstance(x, pd.Timestamp):
                return None
            try:
                val = int(float(x))
                return val if 0 <= val <= 20 else None
            except (ValueError, TypeError):
                return None
        df['duration (years)'] = df['duration (years)'].apply(_clean_years).astype('Int64')

    return df


funding_inscope_data = pd.read_excel(RAW_DATA_DIR / FUNDING_INSCOPE_FILE)
funding_inscope_data = clean_dtypes(funding_inscope_data, has_duration_years=True)
funding_inscope_data.head()

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE OR REPLACE TABLE funding_inscope AS SELECT * FROM funding_inscope_data")
con.close()
print(f"Loaded {len(funding_inscope_data)} rows into {DB_PATH} -> table: funding_inscope")

### 1. Previous year's curated data

In [ ]:
last_year_data = pd.read_excel(RAW_DATA_DIR / LAST_YEAR_FILE)
last_year_data = clean_dtypes(last_year_data, has_duration_years=False)  # no 'duration (years)' column in prior years
last_year_data = assign_stable_row_id(last_year_data, 'lyd_row_id')
last_year_data_edited = last_year_data.copy()
last_year_data.head()

### 2. GFI Grants Tracker data (Airtable export)

In [ ]:
grants_tracker_data = pd.read_excel(RAW_DATA_DIR / GRANTS_TRACKER_FILE)
grants_tracker_data = assign_stable_row_id(grants_tracker_data, 'gt_row_id')  # stamped before filtering, reflects the raw export's row position

grants_tracker_data['EXT_Date added'] = pd.to_datetime(grants_tracker_data['EXT_Date added'], errors='coerce')

europe_filter = (
    (grants_tracker_data['EXT_Funder region'] == 'Europe') |
    (grants_tracker_data['EXT_PI organization region'] == 'Europe')
)
date_filter = (
    (grants_tracker_data['EXT_Date added'] >= f'{PREV_REPORT_YEAR}-01-01') &
    (grants_tracker_data['EXT_Date added'] <= f'{PREV_REPORT_YEAR}-12-31')
)

grants_tracker_data = grants_tracker_data[europe_filter & date_filter].reset_index(drop=True)
print(f"{len(grants_tracker_data)} grants included after filtering")

# Rename Gov't -> Gov throughout the pipeline to avoid apostrophe quoting issues
grants_tracker_data = grants_tracker_data.rename(columns={
    "INT_Gov't contribution (actual currency)": 'INT_Gov contribution (actual currency)',
    "EXT_Gov't contribution (USD)":             'EXT_Gov contribution (USD)',
})

grants_tracker_data_edited = grants_tracker_data.copy()
grants_tracker_data.head()

In [ ]:
valid_countries = set()
for c in pycountry.countries:
    valid_countries.add(c.name.strip().lower())
    if hasattr(c, 'common_name'):
        valid_countries.add(c.common_name.strip().lower())

valid_countries.update({'czech republic', 'russia', 'turkey', 'uk'})

def _is_valid_country_cell(val):
    if pd.isna(val) or str(val).strip() == '':
        return False
    first_part = re.split(r'[;,]', str(val))[0].strip().lower()
    return first_part in valid_countries

before_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
removed_vals = (
    grants_tracker_data['EXT_PI organization country']
    .dropna()
    .loc[lambda s: ~s.apply(_is_valid_country_cell)]
    .unique()
)

for _df in (grants_tracker_data, grants_tracker_data_edited):
    _df['EXT_PI organization country'] = _df['EXT_PI organization country'].apply(
        lambda val: val if _is_valid_country_cell(val) else None
    )

after_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
print(f"Nulled out {before_count - after_count} non-country values ({before_count} -> {after_count} filled)")
print(f"Values removed: {sorted(removed_vals)}")

### 3. Dimensions data (from S1's `{RUN_TABLE}`)

Reads S1's automated Dimensions grants query output directly from `funding.db`, replacing the 4 manually-exported Dimensions Excel files the original prototype used.

**Adapter cell below**: normalizes S1's DSL-native column names/shapes into the flat column names the rest of this notebook expects (matching the legacy manual-export format it was originally built around). If S1's field shapes change, only this cell should need updating - flagged in the implementation plan as needing live validation against a real `dsl.query()` call.

In [ ]:
con = duckdb.connect(str(DB_PATH))
dimensions_raw = con.sql(f"SELECT * FROM {RUN_TABLE}").df()
con.close()
print(f"{len(dimensions_raw)} rows loaded from '{RUN_TABLE}'")

def _join_list(val):
    """Flatten a dimcli-style nested field (list of dicts, list of strings, or a plain
    scalar) into a semicolon-joined string, matching the legacy manual-export format."""
    if val is None or (not isinstance(val, (list, tuple, np.ndarray)) and pd.isna(val)):
        return None
    if isinstance(val, str):
        return val
    if not isinstance(val, (list, tuple, np.ndarray)):
        return str(val)
    parts = []
    for item in val:
        if isinstance(item, dict):
            name = item.get('name')
            if name is None and ('first_name' in item or 'last_name' in item):
                name = f"{item.get('first_name', '')} {item.get('last_name', '')}".strip()
            parts.append(str(name if name is not None else item))
        else:
            parts.append(str(item))
    parts = [p for p in parts if p]
    return '; '.join(parts) if parts else None


def _first_or_none(val):
    joined = _join_list(val)
    if not joined:
        return None
    return joined.split(';')[0].strip()


dimensions_data = pd.DataFrame({
    'Grant ID': dimensions_raw['id'],
    'Title translated': dimensions_raw.get('title'),
    'Title': dimensions_raw.get('original_title'),
    'Abstract translated': dimensions_raw.get('abstract'),
    'Abstract': None,  # not queried by S1 - grants DSL has no separate original-language abstract field
    'Researchers': dimensions_raw['researchers'].apply(_join_list) if 'researchers' in dimensions_raw else None,
    'Research Organization - standardized': dimensions_raw['research_org_names'].apply(_join_list) if 'research_org_names' in dimensions_raw else None,
    # Funding amount/Currency: best-effort mapping pending live validation (flagged in S1's
    # docstring) - the grants DSL exposes converted amounts (funding_usd/eur/gbp) plus
    # funding_currency, not necessarily a clean single "original amount" field like the manual export had.
    'Funding amount': dimensions_raw.get('funding_usd'),
    'Currency': dimensions_raw.get('funding_currency'),
    'Funding amount in USD': dimensions_raw.get('funding_usd'),
    'Start date': dimensions_raw.get('start_date'),
    'Start Year': dimensions_raw.get('start_year'),
    'End Year': pd.to_datetime(dimensions_raw['end_date'], errors='coerce').dt.year if 'end_date' in dimensions_raw else None,
    'State of standardized research organization': None,  # not queried by S1
    'Country of standardized research organization': dimensions_raw['research_org_countries'].apply(_join_list) if 'research_org_countries' in dimensions_raw else None,
    'Funder': dimensions_raw['funder_org_name'].apply(_join_list) if 'funder_org_name' in dimensions_raw else None,
    'Funder Country': dimensions_raw['funder_org_countries'].apply(_join_list) if 'funder_org_countries' in dimensions_raw else None,
    'Source Linkout': dimensions_raw['linkout'].apply(_first_or_none) if 'linkout' in dimensions_raw else None,
})

# Defensive dedup - S1 already dedupes by id, but cheap to re-check across the S1/S2 boundary
before = len(dimensions_data)
dimensions_data = dimensions_data.drop_duplicates(subset='Grant ID').reset_index(drop=True)
print(f"Removed {before - len(dimensions_data)} duplicates ({before} -> {len(dimensions_data)} rows)")

dimensions_data.head()

#### Dimensions -> last year's data, exact ID match

Gap-fills last year's data from Dimensions (never overwrites non-empty cells, except funding columns where an existing 0 is treated as fillable), then removes matched rows from `dimensions_data` - no manual review needed, this is a real business-key match.

In [ ]:
col_map = {
    'Title translated':                              'Title',
    'Title':                                         'Original title',
    'Abstract translated':                           'Abstract',
    'Funding amount':                                'Total amount',
    'Currency':                                      'Currency',
    'Funding amount in USD':                         'Total amount (USD)',
    'Start date':                                    'Project start date',
    'Start Year':                                    'Year project started',
    'End Year':                                      'End date',
    'State of standardized research organization':   'PI organisation state',
    'Country of standardized research organization': 'PI organisation country',
    'Funder':                                        'Funder name',
    'Funder Country':                                'Funder Country',
    'Source Linkout':                                'URL for announcement',
}
funding_cols_lyd = {'Total amount', 'Total amount (USD)'}

lyd_id_map = {}
for idx, val in last_year_data['Identification code'].items():
    if not is_empty(val):
        lyd_id_map.setdefault(str(val).strip(), []).append(idx)

matched_ids = set()
changed_indices = set()
cells_filled = 0

for _, dim_row in dimensions_data.iterrows():
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in lyd_id_map:
        continue
    matched_ids.add(grant_id)
    for lyd_idx in lyd_id_map[grant_id]:
        cells_filled += gap_fill(dim_row, last_year_data_edited, lyd_idx, col_map, funding_cols_lyd, changed_indices)
        cells_filled += gap_fill_researchers_and_orgs(
            dim_row, last_year_data_edited, lyd_idx,
            pi_col='Project lead (PI)', collab_col='Collaborator names',
            org_pi_col='PI organisation', org_collab_col='Collaborator institutions',
            changed_indices=changed_indices,
        )

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Matched {len(matched_ids)} grants with last_year_data")
print(f"Filled {cells_filled} missing values across {len(changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

In [ ]:
export_highlighted_diff(last_year_data, last_year_data_edited, changed_indices,
                         AUDIT_DIR / 'dim_ID_matches_last_year_data.xlsx')

#### Dimensions -> grants tracker, exact ID match

In [ ]:
gt_col_map = {
    'Title translated':                              'EXT_Title',
    'Abstract translated':                           'EXT_Abstract',
    'Funding amount':                                'INT_Total amount (actual currency)',
    'Currency':                                      'INT_Currency type',
    'Funding amount in USD':                         'EXT_Total amount (USD)',
    'Start date':                                    'EXT_Project start date (estimated)',
    'End Year':                                      'INT_End date',
    'State of standardized research organization':   'EXT_PI organisation state',
    'Country of standardized research organization': 'EXT_PI organization country',
    'Funder':                                        'EXT_Funder name',
    'Funder Country':                                'EXT_Funder country',
    'Source Linkout':                                'EXT_URL for announcement',
}
funding_cols_gt = {'INT_Total amount (actual currency)', 'EXT_Total amount (USD)'}

gt_id_map = {}
for idx, val in grants_tracker_data['Dimensions.ai grant ID'].items():
    if not is_empty(val):
        gt_id_map.setdefault(str(val).strip(), []).append(idx)

gt_matched_ids = set()
gt_changed_indices = set()
gt_cells_filled = 0

for _, dim_row in dimensions_data.iterrows():
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in gt_id_map:
        continue
    gt_matched_ids.add(grant_id)
    for gt_idx in gt_id_map[grant_id]:
        gt_cells_filled += gap_fill(dim_row, grants_tracker_data_edited, gt_idx, gt_col_map, funding_cols_gt, gt_changed_indices)
        gt_cells_filled += gap_fill_researchers_and_orgs(
            dim_row, grants_tracker_data_edited, gt_idx,
            pi_col='EXT_Project lead (PI)', collab_col='EXT_Collaborator names',
            org_pi_col='EXT_PI organization', org_collab_col='EXT_Collaborator organizations',
            changed_indices=gt_changed_indices,
        )

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(gt_matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Matched {len(gt_matched_ids)} grants with grants_tracker_data")
print(f"Filled {gt_cells_filled} missing values across {len(gt_changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

In [ ]:
export_highlighted_diff(grants_tracker_data, grants_tracker_data_edited, gt_changed_indices,
                         AUDIT_DIR / 'dim_ID_matches_gt_data.xlsx')

### 4. Grants Tracker vs last year's data
#### a. Dimensions ID match

In [ ]:
last_year_data_pre_gt = last_year_data_edited.copy()

gt_lyd_col_map = {
    'EXT_Title':                              'Title',
    'EXT_Abstract':                           'Abstract',
    'INT_Total amount (actual currency)':     'Total amount',
    'INT_Gov contribution (actual currency)': 'Gov contribution',
    'INT_Currency type':                      'Currency',
    'EXT_Total amount (USD)':                 'Total amount (USD)',
    'EXT_Gov contribution (USD)':             'Gov contribution (USD)',
    'EXT_Funding decision':                   'Funding decision',
    'EXT_Project start date (estimated)':     'Project start date',
    'INT_End date':                           'End date',
    'EXT_Project lead (PI)':                  'Project lead (PI)',
    'EXT_PI organization':                    'PI organisation',
    'EXT_PI organization type':               'PI organisation type',
    'EXT_PI organization country':            'PI organisation country',
    'EXT_PI organization region':              'PI organisation region',
    'PI organisation state':                  'PI organisation state',
    'EXT_Collaborator names':                 'Collaborator names',
    'EXT_Collaborator organizations':         'Collaborator institutions',
    'EXT_Funder name':                        'Funder name',
    'EXT_Funder type':                        'Funder type',
    'EXT_Funder country':                     'Funder Country',
    'EXT_Funder region':                      'Funder region',
    'EXT_Production platform':                'Production platform',
    'EXT_End product type':                   'End product type',
    'EXT_Award purpose':                      'Award purpose',
    'EXT_URL for announcement':               'URL for announcement',
    'EXT_Date added':                         'Date added',
    'EXT_Last modified':                      'Last modified',
    'EXT_Years project starts':               'Year project started',
    'INT_GFI grantee?':                       'GFI grantee',
    'INT_GFI Los?':                            'GFI LOS',
    'INT_Link to Los':                        'Link to LOS',
    'INT_GFI partner?':                       'GFI partner',
    'INT_Tier':                                'Tier',
}
funding_cols_lyd2 = {'Total amount', 'Total amount (USD)', 'Gov contribution', 'Gov contribution (USD)'}

lyd2_id_map = {}
for idx, val in last_year_data_edited['Identification code'].items():
    if not is_empty(val):
        lyd2_id_map.setdefault(str(val).strip(), []).append(idx)

gt_lyd_matched_ids = set()
gt_lyd_changed_indices = set()
gt_lyd_cells_filled = 0

for _, gt_row in grants_tracker_data_edited.iterrows():
    grant_id = str(gt_row['Dimensions.ai grant ID']).strip()
    if grant_id not in lyd2_id_map:
        continue
    gt_lyd_matched_ids.add(grant_id)
    for lyd_idx in lyd2_id_map[grant_id]:
        gt_lyd_cells_filled += gap_fill(gt_row, last_year_data_edited, lyd_idx, gt_lyd_col_map, funding_cols_lyd2, gt_lyd_changed_indices)

before = len(grants_tracker_data_edited)
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['Dimensions.ai grant ID'].astype(str).str.strip().isin(gt_lyd_matched_ids)
].reset_index(drop=True)
after = len(grants_tracker_data_edited)

print(f"Matched {len(gt_lyd_matched_ids)} grants between grants_tracker_data_edited and last_year_data_edited")
print(f"Filled {gt_lyd_cells_filled} missing values across {len(gt_lyd_changed_indices)} rows")
print(f"Removed {before - after} rows from grants_tracker_data_edited ({after} remaining)")

In [ ]:
# OPTIONAL - shows which grants were duplicated within the grants tracker data
dupes = grants_tracker_data[
    grants_tracker_data['Dimensions.ai grant ID'].isin(gt_lyd_matched_ids)
    & grants_tracker_data['Dimensions.ai grant ID'].duplicated(keep=False)
][['Dimensions.ai grant ID', 'EXT_Title']].sort_values('Dimensions.ai grant ID')
dupes

In [ ]:
export_highlighted_diff(last_year_data_pre_gt, last_year_data_edited, gt_lyd_changed_indices,
                         AUDIT_DIR / 'gt_changes_last_year_data.xlsx')

#### b. Title-based match - REVIEW POINT 1 (exact title)

In [ ]:
lyd_title_map = {}
for idx, val in last_year_data_edited['Title'].items():
    norm = normalize_title(val)
    if norm:
        lyd_title_map.setdefault(norm, []).append(idx)

title_match_rows = []
for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    norm = normalize_title(gt_row.get('EXT_Title'))
    if norm and norm in lyd_title_map:
        for lyd_idx in lyd_title_map[norm]:
            title_match_rows.append({
                'gt_row_id':               gt_row['gt_row_id'],
                'lyd_row_id':              last_year_data_edited.at[lyd_idx, 'lyd_row_id'],
                'GT index':                gt_idx,
                'LYD index':               lyd_idx,
                'Title':                   gt_row.get('EXT_Title'),
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
                'GT Funder':               gt_row.get('EXT_Funder name'),
                'LYD Funder':              last_year_data_edited.at[lyd_idx, 'Funder name'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LYD Total (USD)':         last_year_data_edited.at[lyd_idx, 'Total amount (USD)'],
            })

title_matches_df = pd.DataFrame(title_match_rows)
print(f"{len(title_matches_df)} (GT, LYD) pairs found")
title_matches_df

In [ ]:
export_for_review(
    title_matches_df, id_cols=['gt_row_id', 'lyd_row_id'],
    out_path=DATA_DIR / f'gt_title_match_for_review_{RUN_DATE}.csv',
)

**Pause here.** Open the exported CSV in `data/`, review the pre-filled `is_true_match` column (defaults to `True` = confirmed match -> gap-fill + remove), flip any false positives to `False` (kept, falls through to the fuzzy-match step below), save it as `..._reviewed_{date}.csv` in the same folder (the exact filename was printed by the cell above). Then continue.

In [ ]:
last_year_data_pre_title = last_year_data_edited.copy()

confirmed_title, rejected_title = apply_reviewed_decisions(
    title_matches_df, DATA_DIR / f'gt_title_match_reviewed_{RUN_DATE}.csv',
    id_cols=['gt_row_id', 'lyd_row_id'],
)
print(f"{len(confirmed_title)} confirmed, {len(rejected_title)} rejected")

title_changed_indices = set()
title_cells_filled = 0

for _, match in confirmed_title.iterrows():
    gt_idx = int(match['GT index'])
    lyd_idx = int(match['LYD index'])
    gt_row = grants_tracker_data_edited.loc[gt_idx]
    title_cells_filled += gap_fill(gt_row, last_year_data_edited, lyd_idx, gt_lyd_col_map, funding_cols_lyd2, title_changed_indices)

confirmed_gt_row_ids = set(confirmed_title['gt_row_id'])
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['gt_row_id'].isin(confirmed_gt_row_ids)
].reset_index(drop=True)

print(f"Filled {title_cells_filled} missing values across {len(title_changed_indices)} rows")
print(f"Removed {len(confirmed_gt_row_ids)} confirmed matches from grants_tracker_data_edited ({len(grants_tracker_data_edited)} remaining)")

In [ ]:
export_highlighted_diff(last_year_data_pre_title, last_year_data_edited, title_changed_indices,
                         AUDIT_DIR / 'gt_title-match_changes_last_year_data.xlsx')

#### c. Partial title-based match - REVIEW POINT 2 (fuzzy title)

Uses `rapidfuzz.token_sort_ratio`: tokens are sorted before comparing, so word-order differences score highly. Gap-filling is intentionally skipped for partial matches - not reliable enough to fill data across - so a confirmed match here just removes the row (it's a duplicate, already tracked); a rejected one is kept and later appended as new data.

In [ ]:
try:
    from rapidfuzz import fuzz as _rfuzz
except ImportError:
    raise ImportError("rapidfuzz not found - run: conda install -c conda-forge rapidfuzz")

lyd_title_list = [
    (idx, normalize_title(val))
    for idx, val in last_year_data_edited['Title'].items()
    if not is_empty(val)
]

partial_match_rows = []
for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    gt_norm = normalize_title(gt_row.get('EXT_Title'))
    if not gt_norm:
        continue
    for lyd_idx, lyd_norm in lyd_title_list:
        score = _rfuzz.token_sort_ratio(gt_norm, lyd_norm)
        if score >= FUZZY_THRESHOLD:
            partial_match_rows.append({
                'gt_row_id':               gt_row['gt_row_id'],
                'lyd_row_id':              last_year_data_edited.at[lyd_idx, 'lyd_row_id'],
                'GT index':                gt_idx,
                'LYD index':               lyd_idx,
                'Similarity':              score,
                'GT Title':                gt_row.get('EXT_Title'),
                'LYD Title':               last_year_data_edited.at[lyd_idx, 'Title'],
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
            })

partial_matches_df = pd.DataFrame(partial_match_rows).sort_values('Similarity', ascending=False).reset_index(drop=True)
print(f"{len(partial_matches_df)} (GT, LYD) pairs found at threshold {FUZZY_THRESHOLD}")
partial_matches_df

In [ ]:
export_for_review(
    partial_matches_df, id_cols=['gt_row_id', 'lyd_row_id'],
    out_path=DATA_DIR / f'gt_partial_title_match_for_review_{RUN_DATE}.csv',
)

**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed duplicate, remove" here (no gap-fill). Flip false positives to `False` to keep them as new data instead. Save as `..._reviewed_{date}.csv`, then continue.

In [ ]:
confirmed_partial, rejected_partial = apply_reviewed_decisions(
    partial_matches_df, DATA_DIR / f'gt_partial_title_match_reviewed_{RUN_DATE}.csv',
    id_cols=['gt_row_id', 'lyd_row_id'],
)
print(f"{len(confirmed_partial)} confirmed duplicates (removed, no gap-fill), {len(rejected_partial)} rejected (kept as new)")

confirmed_gt_row_ids = set(confirmed_partial['gt_row_id'])
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['gt_row_id'].isin(confirmed_gt_row_ids)
].reset_index(drop=True)
print(f"Removed {len(confirmed_gt_row_ids)} rows from grants_tracker_data_edited ({len(grants_tracker_data_edited)} remaining)")

#### d. Append remaining grants tracker rows to last year's data

Whatever's left in `grants_tracker_data_edited` after all the matching above is genuinely new - append it.

In [ ]:
append_extra_map = {
    'Dimensions.ai grant ID': 'Identification code',
}
full_append_map = {**gt_lyd_col_map, **append_extra_map}

gt_for_append = grants_tracker_data_edited.rename(columns={
    gt_col: lyd_col for gt_col, lyd_col in full_append_map.items()
})
gt_for_append = gt_for_append[[c for c in gt_for_append.columns if c in last_year_data_edited.columns]].copy()
gt_for_append['Database'] = 'airtable'

last_year_data_edited = pd.concat([last_year_data_edited, gt_for_append], ignore_index=True)

print(f"Appended {len(grants_tracker_data_edited)} rows from grants_tracker_data_edited")
print(f"Total rows in last_year_data_edited: {len(last_year_data_edited)}")

### Dimensions title matching
#### a. Full title match - REVIEW POINT 3 (exact title)

Matches on `Title translated` in `dimensions_data` against `Title` in `last_year_data_edited` (now including the grants-tracker rows appended above).

In [ ]:
lyd_dim_title_map = {}
for idx, val in last_year_data_edited['Title'].items():
    norm = normalize_title(val)
    if norm:
        lyd_dim_title_map.setdefault(norm, []).append(idx)

dim_title_match_rows = []
for dim_idx, dim_row in dimensions_data.iterrows():
    norm = normalize_title(dim_row.get('Title translated'))
    if norm and norm in lyd_dim_title_map:
        for lyd_idx in lyd_dim_title_map[norm]:
            dim_title_match_rows.append({
                'Grant ID':                dim_row['Grant ID'],
                'lyd_row_id':              last_year_data_edited.at[lyd_idx, 'lyd_row_id'],
                'Dim index':               dim_idx,
                'LYD index':               lyd_idx,
                'Title':                   dim_row.get('Title translated'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
                'Dim Funder':              dim_row.get('Funder'),
                'LYD Funder':              last_year_data_edited.at[lyd_idx, 'Funder name'],
                'Dim Total (USD)':         dim_row.get('Funding amount in USD'),
                'LYD Total (USD)':         last_year_data_edited.at[lyd_idx, 'Total amount (USD)'],
            })

dim_title_matches_df = pd.DataFrame(dim_title_match_rows)
print(f"{len(dim_title_matches_df)} (Dim, LYD) pairs found")
dim_title_matches_df

In [ ]:
export_for_review(
    dim_title_matches_df, id_cols=['Grant ID', 'lyd_row_id'],
    out_path=DATA_DIR / f'dim_title_match_for_review_{RUN_DATE}.csv',
)

**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed match, gap-fill + remove". Save as `..._reviewed_{date}.csv`, then continue.

In [ ]:
last_year_data_pre_dim_title = last_year_data_edited.copy()

confirmed_dim_title, rejected_dim_title = apply_reviewed_decisions(
    dim_title_matches_df, DATA_DIR / f'dim_title_match_reviewed_{RUN_DATE}.csv',
    id_cols=['Grant ID', 'lyd_row_id'],
)
print(f"{len(confirmed_dim_title)} confirmed, {len(rejected_dim_title)} rejected")

dim_title_changed_indices = set()
dim_title_cells_filled = 0

for _, match in confirmed_dim_title.iterrows():
    dim_idx = int(match['Dim index'])
    lyd_idx = int(match['LYD index'])
    dim_row = dimensions_data.loc[dim_idx]
    dim_title_cells_filled += gap_fill(dim_row, last_year_data_edited, lyd_idx, col_map, funding_cols_lyd, dim_title_changed_indices)

confirmed_grant_ids = set(confirmed_dim_title['Grant ID'].astype(str))
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).isin(confirmed_grant_ids)
].reset_index(drop=True)

print(f"Filled {dim_title_cells_filled} missing values across {len(dim_title_changed_indices)} rows")
print(f"Removed {len(confirmed_grant_ids)} confirmed matches from dimensions_data ({len(dimensions_data)} remaining)")

In [ ]:
export_highlighted_diff(last_year_data_pre_dim_title, last_year_data_edited, dim_title_changed_indices,
                         AUDIT_DIR / 'dim_title_changes_last_year_data.xlsx')

#### b. Partial title match - REVIEW POINT 4 (fuzzy title)

The original prototype found far fewer partial matches here (6) than a prior manual pass reportedly found (73). The diagnostic cell below narrows down *why* - it does not resolve the discrepancy on its own; getting the original methodology to compare against is still needed for a definitive answer.

In [ ]:
print(f"Candidate pool sizes at this point: dimensions_data={len(dimensions_data)}, last_year_data_edited={len(last_year_data_edited)}")

_lyd_titles = [normalize_title(v) for v in last_year_data_edited['Title'] if not is_empty(v)]
_dim_titles_translated = [normalize_title(v) for v in dimensions_data['Title translated'] if not is_empty(v)]
print(f"Populated 'Title translated' in dimensions_data: {len(_dim_titles_translated)}/{len(dimensions_data)}")

print("\nMatch counts by threshold (token_sort_ratio):")
for threshold in (85, 80, 75, 70):
    count = sum(
        1
        for dim_norm in _dim_titles_translated
        for lyd_norm in _lyd_titles
        if _rfuzz.token_sort_ratio(dim_norm, lyd_norm) >= threshold
    )
    print(f"  threshold {threshold}: {count} (Dim, LYD) pairs")

print(f"\nMatch counts by scorer (threshold={FUZZY_THRESHOLD}):")
for scorer_name, scorer in (('token_sort_ratio', _rfuzz.token_sort_ratio),
                             ('partial_ratio', _rfuzz.partial_ratio),
                             ('WRatio', _rfuzz.WRatio)):
    count = sum(
        1
        for dim_norm in _dim_titles_translated
        for lyd_norm in _lyd_titles
        if scorer(dim_norm, lyd_norm) >= FUZZY_THRESHOLD
    )
    print(f"  {scorer_name}: {count} (Dim, LYD) pairs")

In [ ]:
lyd_dim_partial_list = [
    (idx, normalize_title(val))
    for idx, val in last_year_data_edited['Title'].items()
    if not is_empty(val)
]

dim_partial_match_rows = []
for dim_idx, dim_row in dimensions_data.iterrows():
    dim_norm = normalize_title(dim_row.get('Title translated'))
    if not dim_norm:
        continue
    for lyd_idx, lyd_norm in lyd_dim_partial_list:
        score = _rfuzz.token_sort_ratio(dim_norm, lyd_norm)
        if score >= FUZZY_THRESHOLD:
            dim_partial_match_rows.append({
                'Grant ID':                dim_row['Grant ID'],
                'lyd_row_id':              last_year_data_edited.at[lyd_idx, 'lyd_row_id'],
                'Dim index':               dim_idx,
                'LYD index':               lyd_idx,
                'Similarity':              score,
                'Dim Title':               dim_row.get('Title translated'),
                'LYD Title':               last_year_data_edited.at[lyd_idx, 'Title'],
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
            })

dim_partial_matches_df = pd.DataFrame(dim_partial_match_rows).sort_values('Similarity', ascending=False).reset_index(drop=True)
print(f"{len(dim_partial_matches_df)} (Dim, LYD) pairs found at threshold {FUZZY_THRESHOLD}")
dim_partial_matches_df

In [ ]:
export_for_review(
    dim_partial_matches_df, id_cols=['Grant ID', 'lyd_row_id'],
    out_path=DATA_DIR / f'dim_partial_title_match_for_review_{RUN_DATE}.csv',
)

**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed duplicate, remove" (no gap-fill). Save as `..._reviewed_{date}.csv`, then continue.

In [ ]:
confirmed_dim_partial, rejected_dim_partial = apply_reviewed_decisions(
    dim_partial_matches_df, DATA_DIR / f'dim_partial_title_match_reviewed_{RUN_DATE}.csv',
    id_cols=['Grant ID', 'lyd_row_id'],
)
print(f"{len(confirmed_dim_partial)} confirmed duplicates (removed, no gap-fill), {len(rejected_dim_partial)} rejected (kept as new)")

confirmed_grant_ids = set(confirmed_dim_partial['Grant ID'].astype(str))
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).isin(confirmed_grant_ids)
].reset_index(drop=True)
print(f"Removed {len(confirmed_grant_ids)} rows from dimensions_data ({len(dimensions_data)} remaining)")

### Final outputs

Both written as DuckDB tables in `funding.db` (the programmatic contract S3 needs, since it's an automated script, not a notebook), plus Excel mirrors kept purely for human inspection.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE TABLE {DEDUP_TABLE} AS SELECT * FROM dimensions_data")
con.execute(f"CREATE OR REPLACE TABLE {CURATED_TABLE} AS SELECT * FROM last_year_data_edited")
con.close()

print(f"Saved {len(dimensions_data)} new/untracked grants -> table '{DEDUP_TABLE}' (S3's input)")
print(f"Saved {len(last_year_data_edited)} curated rows -> table '{CURATED_TABLE}' (archival record)")

# Human-readable Excel mirrors, not re-read by anything
dimensions_data.to_excel(AUDIT_DIR / f'{DEDUP_TABLE}.xlsx', index=False)
last_year_data_edited.to_excel(AUDIT_DIR / f'{CURATED_TABLE}.xlsx', index=False)
print(f"Excel copies saved to {AUDIT_DIR}/")